# Experiment 1: geometric measurement of every puff, compared with the lab (zero training)

**CIS 831 Deep Learning project, Kansas State University**

This notebook runs **Experiment 1** on all trial videos.

**Nothing is learned from the lab data.** The millimetres come only from the die-plate ruler (holes A and B, 55 mm apart), so every clip is a test clip and the lab measurements are used only to score the result.

| Step | What happens | Saved, so it is done only once |
|---|---|---|
| 1–2 | Settings, and the lab measurements (thread length + caliper diameter, 5 pieces per clip now, 10 later) | — |
| 3–4 | List the videos; make a "clean die" picture of each | `results/<video>/clean_die.png` |
| 5 | **You click** hole A, hole B and the visible bolt(s) once per video | `results/<video>/clicks.json` |
| 6 | Exact hole centres from the rope ends; pixel → mm scale | added to `clicks.json` |
| 7–9 | SAM 2.1 finds and tracks every piece (RED = cut moment, BLUE = just detached) | `results/<video>/cuts.csv`, `masks/` |
| 10–12 | Every RED and BLUE mask is measured (length along the bend, median diameter), explained with figures | `results/<video>/pieces.csv` |
| 13–14 | Bad masks are dropped (ragged, merged, cut off, unstable), explained with figures | — |
| 15–18 | Clip averages vs lab for every clip: errors, noise floor, agreement plots | `results/experiment1/` |

**Re-running is cheap.** Clicks, tracking and measurements are cached per video. A new video only needs its own clicks and tracking, and changing a drop limit only re-runs the last steps.

### How to run (VS Code)
- Select the kernel **Python (extruder-sam2)** and run the cells **one at a time** (Shift+Enter).
- **Step 5** is interactive. Click the points for every video and press **Save & next**; then continue.
- **Time:** the first full run takes about 2–3 hours (mostly SAM 2 tracking, about 1 s per cut). A re-run takes minutes.

## Step 1: Imports and settings

In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

from extruder import video as vid, calib, measure, pipeline, viz, track, labels as labmod, explain, experiment
from extruder.ui import MultiVideoClicker

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ---- folders -----------------------------------------------------------------------------
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
VIDEO_DIR = DATA_DIR / "Video from Trial Run 10.07.26"     # the trial videos (901_1.MP4 ...)
EXCLUDE = ("Blank",)                                         # videos whose name contains these are skipped
RESULTS_DIR = PROJECT_DIR / "results"                        # one folder per video (cached work)
EXP_DIR = RESULTS_DIR / "experiment1"                        # the experiment's tables and figures

# ---- lab measurements ------------------------------------------------------------------------
LABELS_FILE = DATA_DIR / "Trial Run Data.xlsx"   # .xlsx or .csv, one row per measured piece
LABELS_SHEET = "Oct 7 Final"                      # sheet name (ignored for .csv)
LABEL_COLUMNS = dict(clip="Sl. No.", piece="Piece no.",
                     length="Length of the Product (mm)", diameter="Diameter of the Product (mm)")

# ---- SAM 2.1 ----------------------------------------------------------------------------------
SAM2_CHECKPOINT = PROJECT_DIR / "checkpoints" / "sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- tracking ---------------------------------------------------------------------------------
CROP_SIZE = 360        # square crop around hole A (px)
N_TRACK = None         # None = automatic: about 1.6 knife cycles (17 frames at 450 rpm, 8 at 1005 rpm)
MAX_CUTS = None        # None = every cut; a number = only the first N cuts per video (quick test)
FORCE_TRACKING = False # True = track again even if saved results exist
USE_BRIGHTNESS = False # True = SAM 2 sees each video at the brightness saved in Step 5 (see Step 5)

# ---- measuring and dropping --------------------------------------------------------------------
N_JOBS = max(1, (os.cpu_count() or 2) - 2)   # CPU processes used to measure masks
FORCE_MEASURE = False
measure.MAX_ROUGHNESS_PCT = 2.5              # Check 1: ragged outline above this % -> drop
measure.TUBE_FILL_RANGE = (0.80, 1.20)       # Check 2: area / (length x diameter) outside -> drop
measure.MAX_STABILITY_PCT = 5.0              # Check 3: change between 4 turned copies above this % -> drop

# ---- figures ------------------------------------------------------------------------------------
EXAMPLE_VIDEO = None                         # video used for the explanation figures; None = first one
SEED = 0

EXP_DIR.mkdir(parents=True, exist_ok=True); (EXP_DIR / "figures").mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
FIG = EXP_DIR / "figures"

## Step 2: The lab measurements
One row per measured piece:
- **length** = a thread laid along the piece, straightened and measured;
- **diameter** = calipers.

Per clip we keep the mean, the SD, the number of pieces and the **standard error** (SD / √n), which says how precisely 5 pieces know their own clip's mean. The median standard error over the clips is the **noise floor**: no method can be judged more finely than that.

In [ ]:
lab_pieces = labmod.load_lab_pieces(LABELS_FILE, LABELS_SHEET, LABEL_COLUMNS)
labels = labmod.clip_labels(lab_pieces)
floor = labmod.noise_floor(labels)
print(f"{len(lab_pieces)} measured pieces in {labels['clip'].nunique()} clips "
      f"({lab_pieces.groupby('clip').size().min()}-{lab_pieces.groupby('clip').size().max()} pieces per clip)")
print(f"Noise floor (median standard error of a clip mean): diameter {floor['diameter']:.2f} mm, "
      f"length {floor['length']:.2f} mm")
display(labels.round(2))

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for a, t in zip(ax, ("diameter", "length")):
    for i, (c, g) in enumerate(lab_pieces.groupby("clip")):
        a.scatter(np.full(len(g), i), g[f"{t}_mm"], s=14, color=experiment.SET_COLOURS.get(c[:3], "k"))
    a.set_xticks(range(labels["clip"].nunique())); a.set_xticklabels(sorted(lab_pieces["clip"].unique()), rotation=70, fontsize=8)
    a.set(ylabel=f"{t} (mm)", title=f"Lab {t}: every measured piece (colour = setting)"); a.grid(alpha=0.3)
plt.tight_layout(); fig.savefig(FIG / "lab_pieces.png", dpi=90); plt.show()

## Step 3: The videos
A video belongs to a clip of the lab sheet when its file name (without `.MP4`) equals the clip name, e.g. `901_1.MP4` ↔ `901_1`. Videos without lab measurements are still processed, but they are left out of the comparison.

In [ ]:
videos = [v for v in vid.list_videos(VIDEO_DIR) if not any(x in v.name for x in EXCLUDE)]
_info = {}
def info_of(v):
    v = str(v)
    if v not in _info:
        _info[v] = vid.probe(v)
    return _info[v]

def clip_of(video_stem):
    return video_stem      # the file name is the clip name

def status():
    rows = []
    for v in videos:
        d = RESULTS_DIR / v.stem
        clicks = pipeline.load_clicks(d / "clicks.json")
        rows.append(dict(video=v.name, lab_pieces=int((lab_pieces["clip"] == clip_of(v.stem)).sum()),
                         clean_die=(d / "clean_die.png").exists(), clicked=bool(clicks.get("clicked", {}).get("hole_A")),
                         refined="refined" in clicks, tracked=(d / "tracking.json").exists(),
                         measured=(d / "pieces.csv").exists()))
    return pd.DataFrame(rows)

print(f"{len(videos)} videos in {VIDEO_DIR}")
display(status())

## Step 4: A "clean die" picture of every video
For each pixel we take the median over about 70 frames. Moving product mostly disappears and the die plate stays, which makes the holes and bolts easy to click. This takes about 15 s per video and is saved.

In [ ]:
for v in videos:
    p = RESULTS_DIR / v.stem / "clean_die.png"
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        Image.fromarray(vid.clean_die_image(v)).save(p)
        print(f"{v.name}: clean die picture made in {time.time() - t0:.0f} s")
ncol = 5
fig, ax = plt.subplots(int(np.ceil(len(videos) / ncol)), ncol, figsize=(3.4 * ncol, 2.2 * np.ceil(len(videos) / ncol)), squeeze=False)
for a in ax.ravel(): a.axis("off")
for a, v in zip(ax.ravel(), videos):
    a.imshow(viz.brighten(np.asarray(Image.open(RESULTS_DIR / v.stem / "clean_die.png"))[150:750, 450:1250], 1.6))
    a.set_title(v.stem, fontsize=9)
plt.suptitle("Clean die pictures (brightened, around the die)"); plt.tight_layout(); plt.show()

## Step 5: Click the reference points (once per video)
An interactive picture appears below. For each video:

1. **hole A**: the lower die hole, where the pieces we follow come out (in the trial videos, the lower yellow spot);
2. **hole B**: the upper die hole (the upper yellow spot);
3. **bolt(s)**: click the centre of any bolt head you can see. In the trial videos usually only the **other bolt** (dark circle to the right of the holes) is visible; click "bolt just below hole A" only if you can really see it.

**Brightness** (slider, or **Auto brightness**). Brightens dark videos so the holes and bolts are easier to see; it is saved per video with the points.
- With `USE_BRIGHTNESS = True` (Step 1), SAM 2 also sees the video this bright.
- It is off by default. On the darkest trial video, brightening did not give SAM 2 cleaner masks (slightly more were dropped), and the masks became about 3 % larger in area, which shifts the measured sizes. A hand-picked brightness per video would add a person-dependent setting to the measurement.
- Leave it off unless a video is too dark for SAM 2 to find the pieces at all.

How it works:
- Use the toolbar **magnifier** to zoom in for exact clicks. Click it again to switch zoom off, because clicks are ignored while zoom is on.
- **Copy points from previous video** starts from the last video's points. The camera usually did not move within a setting, so check and fix only what moved.
- **Save & next** saves and jumps to the next video without points. A check mark in the list = saved.
- Clicks are saved in `results/<video>/clicks.json` and are **never asked for again**. Only new videos appear without a check mark.

**How the clicks are used.**
- **Holes A and B set the millimetre scale** (55 mm apart). Both are moved to their exact centres from their rope ends.
- **The bolt is used as a check.**
- If **both** bolts are visible, the bolts set the scale instead (65 mm apart).

In [ ]:
%matplotlib widget
clicker = MultiVideoClicker(videos, RESULTS_DIR, info_of)
clicker.show()

In [ ]:
%matplotlib inline
plt.close(clicker.fig)
st = clicker.status()
display(st)
missing = st.loc[st["ready"] != "yes", "video"].tolist()
if missing:
    print("Still to click:", missing, "- go back to Step 5. Videos without points are skipped below.")
ready = [v for v in videos if clicker.has_clicks(v)]
print(f"{len(ready)} of {len(videos)} videos are ready.")

## Step 6: Exact hole centres and the millimetre scale
- **Holes:** for each video, holes A and B are moved to the median centre of all their fresh rope ends, so a click that is a few pixels off does not matter. This is done once and saved in `clicks.json` (`refined`).
- **Scale:** the pixel → mm scale then comes from the two holes (or the two bolts, if both were clicked).
- **Check:** every other clicked point is compared with where the die-plate map puts it (the "check" distance in mm).

In [ ]:
mappings = {}
for v in ready:
    p = RESULTS_DIR / v.stem / "clicks.json"
    c = json.loads(p.read_text())
    if "refined" not in c:
        refined, _ = pipeline.refine_points(v, info_of(v), c["clicked"], log=lambda s: print(f"  {v.stem}: {s}"))
        c["refined"] = {k: list(map(float, xy)) for k, xy in refined.items()}
        p.write_text(json.dumps(c, indent=1))
    mappings[v.stem] = experiment.video_mapping(c)

rows = []
for v in ready:
    m, name = mappings[v.stem]
    rows.append(dict(video=v.stem, reference=name, px_per_mm=m["px_per_mm"] if m else np.nan,
                     **{f"check_{k}_mm": round(d, 2) for k, d in (m["offset_mm"].items() if m else [])
                        if k not in m["points_used"]}))
ref_table = pd.DataFrame(rows)
display(ref_table.round(3))
explain.reference_grid(ready, RESULTS_DIR, mappings, path=FIG / "reference_points.png"); plt.show()

## Step 7: Load the SAM 2.1 model
**SAM 2** ("Segment Anything Model 2", Meta AI) is a large pretrained neural network that outlines and tracks objects in videos. We use it **as it is, without any training**.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=DEVICE)
track.quiet_sam2()
print(f"SAM 2.1 large loaded on {DEVICE}: {sum(p.numel() for p in predictor.parameters()) / 1e6:.0f} M parameters")

## Step 8: Find and track every piece in every video
For each video, as in notebook 01:
1. crop 360 × 360 px around hole A;
2. find every knife cut from the yellowness "blink";
3. put an automatic box on each fresh rope end;
4. SAM 2 follows the piece until it leaves the hole: **RED** = cut moment, **BLUE** = first detached frame.

**The tracking window adapts to the knife speed:** about 1.6 knife cycles (17 frames at 450 rpm, 10 at 750, 8 at 1005). The tracking checks of Part 1 (never detached, lost, out of the picture, size jump…) are applied here.

**Saved:** a video that was already tracked with the same settings is skipped.

In [ ]:
track_rows = []
preps = {}
for i, v in enumerate(ready):
    out = RESULTS_DIR / v.stem
    c = json.loads((out / "clicks.json").read_text())
    hole = c["refined"]["hole_A"]
    bright = float(c.get("brightness", 1.0)) if USE_BRIGHTNESS else 1.0
    print(f"[{i + 1}/{len(ready)}] {v.name}")
    t0 = time.time()
    table, prep = pipeline.track_video_cached(predictor, v, out, hole, CROP_SIZE, N_TRACK, MAX_CUTS, DEVICE,
                                              force=FORCE_TRACKING, log=lambda s: print("   ", s), brightness=bright)
    preps[v.stem] = prep
    knife = lab_pieces.loc[lab_pieces["clip"] == clip_of(v.stem), "Knife Speed, rpm"]
    track_rows.append(dict(video=v.stem, cuts_found=len(table), kept_by_tracking=int(table["keep"].sum()),
                           cut_rate_video=round(prep["cuts"]["cut_hz"], 2),
                           cut_rate_from_knife=round(knife.iloc[0] * 3 / 60, 2) if len(knife) else np.nan,
                           window=json.loads((out / "tracking.json").read_text())["n_track"], brightness=bright,
                           minutes=round((time.time() - t0) / 60, 1)))
track_table = pd.DataFrame(track_rows)
display(track_table)

In [ ]:
reasons = []
for v in ready:
    t = pd.read_csv(RESULTS_DIR / v.stem / "cuts.csv")
    reasons.append(t["reason"].fillna("").replace("", "kept").value_counts().rename(v.stem))
print("Tracking checks per video (number of cuts):")
display(pd.DataFrame(reasons).fillna(0).astype(int))

## Step 9: Look at the tracking (one cut per setting)
A quick visual check that RED/BLUE behave at every knife speed: green = attached, **RED** = cut moment, **BLUE** = detached. Run the cell again to see other cuts.

In [ ]:
shown = set()
for v in ready:
    s = clip_of(v.stem).split("_")[0]
    if s in shown:
        continue
    shown.add(s)
    t = pd.read_csv(RESULTS_DIR / v.stem / "cuts.csv")
    k = int(random.choice(t.loc[t["keep"], "cut"].tolist()))
    prep = preps[v.stem]
    n = json.loads((RESULTS_DIR / v.stem / "tracking.json").read_text())["n_track"]
    row, res = pipeline.process_cut(predictor, prep, k, prep["cuts"]["events"][k], device=DEVICE, n_track=n)
    viz.track_strip(prep["frame_files"], res, row["headon_frame"], prep["hole_in_crop"],
                    title=f"{v.stem} (setting {s}), cut {k}: green -> RED (cut) -> BLUE (detached)",
                    path=FIG / f"tracking_example_{s}.png")
    plt.show()

## Step 10: Measure every RED and BLUE mask
Each mask is turned into millimetres with the video's scale and measured:
- **length** along the bend (like the lab's thread);
- **diameter** = median diameter of the biggest circles that fit inside it.

Every piece is measured 4 times, turned by 0/45/90/135°, and the median is kept. The three drop scores are computed at the same time.

This runs on several CPU processes. It takes about 1–2 minutes per video and is saved in `results/<video>/pieces.csv`.

In [ ]:
per_video = {}
for i, v in enumerate(ready):
    m, name = mappings[v.stem]
    px_to_mm = m["px_to_mm"] if m else experiment.nominal_mapping()
    ppm = m["px_per_mm"] if m else experiment.NOMINAL_PX_PER_MM
    t0 = time.time()
    per_video[v.stem] = experiment.measure_video(RESULTS_DIR / v.stem, px_to_mm, ppm, name, n_jobs=N_JOBS,
                                                 force=FORCE_MEASURE)
    print(f"[{i + 1}/{len(ready)}] {v.stem}: {len(per_video[v.stem])} masks ({time.time() - t0:.0f} s)")
pieces = experiment.collect_pieces(per_video, clip_of)
print(f"\n{len(pieces)} masks measured in {pieces['video'].nunique()} videos "
      f"({(pieces['key'] == 'red').sum()} RED, {(pieces['key'] == 'blue').sum()} BLUE)")

## Step 11: How one piece is measured
The same explanation as in `reports/how_we_measure_a_puff.pdf`, on a real piece of this experiment (a BLUE mask of the example video, with a typical length):
1. the mask, then in millimetres;
2. the two tips, found by walking inside the piece;
3. the "thread" along the ridge of the biggest inside circles;
4. **length** = between the last full-size circles + one radius at each end;
5. **diameter** = median circle diameter over the central 60 %.

Run the cell again for another piece.

In [ ]:
ex_video = EXAMPLE_VIDEO or ready[0].stem
ex_map = mappings[ex_video][0]["px_to_mm"]
ok = pieces[(pieces["video"] == ex_video) & (pieces["key"] == "blue") & pieces["used"]]
typical = ok.iloc[(ok["length_mm"] - ok["length_mm"].median()).abs().argsort()[:10]]
ex_cut = int(typical.sample(1, random_state=None)["cut"].iloc[0])
explain.measure_steps(RESULTS_DIR / ex_video, ex_cut, "blue", ex_map, path=FIG / "how_one_piece_is_measured.png")
plt.show()

## Step 12: Why the median diameter?
Each piece gives hundreds of circle diameters. The **median over the central 60 %** is the typical thickness of the body of the piece:
- the **biggest** circle picks bulges at bends;
- the **smallest** picks pinches and bad edge pixels;
- the **average** is pulled by both.

The biggest and the average are saved too (`width_max_mm`, `width_mean_mm`). The table compares them over all used pieces.

In [ ]:
u = pieces[pieces["used"]]
display(u.groupby("key")[["width_mm", "width_mean_mm", "width_max_mm", "length_mm", "feret_mm"]].mean().round(2)
        .rename(columns={"width_mm": "diameter (median, used)", "width_mean_mm": "area/length",
                         "width_max_mm": "biggest circle", "length_mm": "length along bend", "feret_mm": "straight length"}))

## Step 13: How bad masks are dropped
Same explanation as in `reports/how_we_drop_bad_masks.pdf`, on masks of the example video:
- **Check 1, ragged outline:** compare the mask with a smoothed copy of itself; more than 2.5 % of the area changing = ragged.
- **Check 2, one tube?** area ÷ (length × diameter). Above 1.20 = another piece merged in; below 0.80 = part of the piece missing.
- **Check 3, direction:** the 4 turned copies disagree by more than 5 % = the measurement is unsure.

In [ ]:
ev = pieces[(pieces["video"] == ex_video) & (pieces["key"] == "blue")]
clean_cut = int(ev[ev["used"]].sample(1, random_state=SEED)["cut"].iloc[0])
ragged = ev[ev["mask_problem"] == "ragged_outline"].sort_values("roughness_pct", ascending=False)["cut"].head(2).tolist()
explain.roughness_check(RESULTS_DIR / ex_video, [clean_cut] + [int(c) for c in ragged], "blue", ex_map,
                        path=FIG / "drop_check1_roughness.png"); plt.show()
odd = ev[ev["mask_problem"].isin(["extra_area_merged_piece", "missing_area_cut_off_piece"])]["cut"].head(2).tolist()
explain.tube_check(RESULTS_DIR / ex_video, [clean_cut] + [int(c) for c in odd], "blue", ex_map,
                   path=FIG / "drop_check2_tube.png"); plt.show()

In [ ]:
for key in experiment.KEYS:
    explain.checks_histograms(pieces, key, title=f"All {key.upper()} masks of all videos",
                              path=FIG / f"drop_scores_{key}.png"); plt.show()
explain.dropped_gallery(pieces, RESULTS_DIR, ex_video, "blue", path=FIG / "dropped_gallery_example.png"); plt.show()

## Step 14: How many masks each check dropped, per clip

In [ ]:
drop_table = (pieces.assign(result=pieces["mask_problem"].replace("", "used"))
              .pivot_table(index=["clip", "key"], columns="result", values="cut", aggfunc="count", fill_value=0))
display(drop_table)
print("Overall:"); display(pieces.assign(result=pieces["mask_problem"].replace("", "used"))
                           .groupby(["key", "result"]).size().unstack(0))

## Step 15: Clip averages: video vs lab
For every clip and mask type: the number of pieces used, and the mean ± SD of diameter and length.

The plot shows the video values (ruler only, nothing learned) next to the lab's mean ± SD.

In [ ]:
ct = experiment.clip_table(pieces, labels)
show_cols = ["clip", "key", "n_measured", "n_used", "reference", "px_per_mm",
             "video_diameter_mean", "video_diameter_sd", "lab_diameter_mean", "lab_diameter_sd",
             "video_length_mean", "video_length_sd", "lab_length_mean", "lab_length_sd"]
display(ct[show_cols].round(2))
for key in experiment.KEYS:
    experiment.plot_clips(ct, key, path=FIG / f"clips_{key}.png"); plt.show()

## Step 16: Video vs lab, every clip (zero training)
For every clip, the video's mean diameter and mean length are compared with the lab's means.

| Measure | Meaning |
|---|---|
| **MAE** | average size of the error per clip (mm), with a 95 % bootstrap interval |
| RMSE | like MAE, but big misses count more |
| MAPE | error in % of the lab value |
| **bias** | average signed error: is the video always too big or too small? |
| limits_of_agreement | ± range that holds 95 % of the clip errors around the bias (Bland–Altman) |
| R², correlation | how well the clip-to-clip differences are followed. Meaningful for length; diameter hardly changes between clips |
| **noise_floor** | the lab's own precision for a clip mean (5 pieces). An MAE near this is as good as the data can show |
| SD_MAE | how well the video's spread (SD) matches the lab's spread |

Two variants: **bad masks dropped** (the main result) and **every mask kept** (to see what the shape checks are worth).

In [ ]:
comp = experiment.compare(ct)
met = experiment.metrics(comp)
cols = ["variant", "key", "target", "clips", "MAE", "MAE_CI_low", "MAE_CI_high", "RMSE", "MAPE", "bias",
        "limits_of_agreement", "R2", "correlation", "noise_floor", "SD_MAE"]
display(met[cols].sort_values(["target", "variant", "key"]).round(3))
experiment.plot_errors(met, path=FIG / "errors.png"); plt.show()

## Step 17: Agreement plots
- **Left:** video vs lab, one point per clip, coloured by setting (bars = the lab's standard error). On the dashed line, the video and the lab agree exactly.
- **Right:** Bland–Altman plot: difference (video − lab) against the average, with the bias (solid line) and the 95 % limits of agreement (dashed). This is the standard way in food science to compare two measuring methods.

In [ ]:
for key in experiment.KEYS:
    experiment.plot_agreement(comp, key, path=FIG / f"agreement_{key}.png"); plt.show()

## Step 18: Errors clip by clip, and per setting
Video − lab for every clip (main variant), so that clips or settings that go wrong are easy to spot.

In [ ]:
main = comp[comp["variant"] == experiment.MAIN_VARIANT]
by_clip = main.pivot_table(index=["setting", "clip"], columns=["target", "key"], values="error")
display(by_clip.round(2))
print("Per setting (mean video − lab, mm):")
display(main.pivot_table(index="setting", columns=["target", "key"], values="error", aggfunc="mean").round(2))

## Step 19: Save everything
| File | Content |
|---|---|
| `experiment1/pieces.csv` | **one row per measured mask**: clip, cut, RED/BLUE, frame, length, diameter (+ thickest, area / length, straight length, area), drop scores and reason. Use this file to compute the expansion ratio with your formulas |
| `experiment1/clip_table.csv` | per clip and mask type: pieces used, video mean / SD / median, lab mean / SD / n / standard error |
| `experiment1/comparison.csv` | per clip, mask type, target and variant: video mean, lab mean, error |
| `experiment1/metrics.csv` | MAE (+ 95 % interval), RMSE, MAPE, bias, limits of agreement, R², correlation, noise floor, SD error |
| `experiment1/references.csv`, `tracking.csv` | scale per video; cuts found / kept and cut rate per video |
| `experiment1/settings.json` | every setting of this run |
| `experiment1/figures/` | all figures above |

In [ ]:
keep_cols = ["video", "clip", "setting", "cut", "key", "frame", "frames_to_cut", "used", "mask_problem",
             "length_mm", "width_mm", "width_max_mm", "width_mean_mm", "feret_mm", "area_mm2", "straightness",
             "roughness_pct", "tube_fill", "shape_stability_pct", "px_per_mm", "reference", "mask_file"]
pieces[[c for c in keep_cols if c in pieces]].to_csv(EXP_DIR / "pieces.csv", index=False)
ct.to_csv(EXP_DIR / "clip_table.csv", index=False)
comp.to_csv(EXP_DIR / "comparison.csv", index=False)
met.to_csv(EXP_DIR / "metrics.csv", index=False)
ref_table.to_csv(EXP_DIR / "references.csv", index=False)
track_table.to_csv(EXP_DIR / "tracking.csv", index=False)
settings = dict(videos=[v.name for v in ready], labels_file=str(LABELS_FILE), labels_sheet=LABELS_SHEET,
                crop_size=CROP_SIZE, n_track=N_TRACK, max_cuts=MAX_CUTS, use_brightness=USE_BRIGHTNESS,
                max_roughness_pct=measure.MAX_ROUGHNESS_PCT, tube_fill_range=list(measure.TUBE_FILL_RANGE),
                max_stability_pct=measure.MAX_STABILITY_PCT, tracking_version=pipeline.TRACKING_VERSION,
                measure_version=experiment.MEASURE_VERSION, sam2_checkpoint=SAM2_CHECKPOINT.name)
(EXP_DIR / "settings.json").write_text(json.dumps(settings, indent=1))
for f in sorted(EXP_DIR.glob("*")):
    print(f.relative_to(RESULTS_DIR))

## Notes
- **Zero training:** nothing in this notebook is fitted to the lab data. The lab values are used only for scoring.
- **Re-running:** clicks, refined points, tracking and measurements are saved per video. Running the notebook again only re-does the comparison (Steps 15–19), plus the work for any new video.
  - `FORCE_TRACKING` / `FORCE_MEASURE` force a redo.
  - Changing a drop limit in Step 1 only needs Steps 10 onward, and the masks are not re-measured.
- **New videos:** put them in `VIDEO_DIR` and name them like the clip in the lab sheet. They show up in Step 5 without a check mark.
- **More lab pieces (10 per clip):** replace the sheet. Nothing else changes; the noise floor will drop.
- **Limits** of the drop checks were set by eye on one clip and still need confirming on a small hand-labelled set.